# Notebook 03: Enriquecimiento espacial con comisarías y franjas de distancia

**Proyecto:** Análisis de delitos geolocalizados en CABA  
**Dataset principal:** `delitos_2023_caba_limpio.csv`  
**Sedes policiales:** `comisarias_policia(1).csv`  
**Jurisdicciones:** `division_comisaria_vecinal.csv`

## Objetivo

Enriquecer el dataset limpio con información espacial útil para un panel de control:

1. identificar la comisaría vecinal con jurisdicción sobre cada hecho;
2. identificar la sede policial físicamente más cercana;
3. calcular la distancia en metros hasta esa sede;
4. convertir la distancia numérica en una franja categórica interpretable.

## Decisión metodológica: se utilizan ambos datasets

Los dos archivos de comisarías no son sustitutos:

- el archivo de **polígonos** responde qué comisaría vecinal tiene jurisdicción territorial;
- el archivo de **puntos** responde dónde está físicamente cada sede policial y permite calcular distancias.

Por eso se combinan, pero cada uno cumple una función específica.

## Columnas nuevas del CSV final

Se agregan únicamente cuatro variables:

| Columna | Utilidad en el panel |
|---|---|
| `comisaria_jurisdiccional` | Filtrar y comparar hechos por jurisdicción |
| `comisaria_mas_cercana` | Identificar la sede policial físicamente más próxima |
| `distancia_comisaria_metros` | Usar la distancia como medida numérica |
| `franja_distancia_comisaria` | Usar la distancia como categoría simple |

No se agregan columnas redundantes como comuna policial, código de división o método de asignación. Esos controles quedan documentados dentro del notebook, pero no cargan innecesariamente el dataset final.

## 1. Dependencias

Además de las librerías habituales del proyecto, este notebook utiliza `geopandas`, `shapely` y `pyproj` para trabajar con polígonos.

Si aparece un error indicando que falta alguna librería, ejecutar una sola vez en la terminal del entorno virtual:

```powershell
python -m pip install geopandas shapely pyproj
```

In [ ]:
import sys

print(sys.executable)

c:\Users\nikko\CienciaDatosDelitosTPO\.venv-1\Scripts\python.exe


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

ModuleNotFoundError: No module named 'geopandas'

## 2. Rutas del proyecto

El código detecta si el notebook se ejecuta desde la carpeta `notebooks` o desde la raíz del repositorio.

Estructura recomendada:

```text
CienciaDatosDelitosTPO/
├── datasets/
│   ├── raw/
│   │   ├── comisarias_policia(1).csv
│   │   └── division_comisaria_vecinal.csv
│   └── processed/
│       └── delitos_2023_caba_limpio.csv
├── notebooks/
│   └── 03_enriquecimiento_comisarias_y_franjas.ipynb
└── reports/
    ├── figures/
    └── tables/
```

In [ ]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "datasets" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "datasets" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

crime_candidates = [
    PROCESSED_DATA_DIR / "delitos_2023_caba_limpio.csv",
    PROCESSED_DATA_DIR / "delitos_caba_limpio.csv",
    PROJECT_ROOT / "data" / "processed" / "delitos_2023_caba_limpio.csv",
    Path("/mnt/data/delitos_2023_caba_limpio.csv"),
]

station_candidates = [
    RAW_DATA_DIR / "comisarias_policia(1).csv",
    RAW_DATA_DIR / "comisarias_policia.csv",
    PROJECT_ROOT / "comisarias_policia(1).csv",
    Path("/mnt/data/comisarias_policia(1).csv"),
]

jurisdiction_candidates = [
    RAW_DATA_DIR / "division_comisaria_vecinal.csv",
    PROJECT_ROOT / "division_comisaria_vecinal.csv",
    Path("/mnt/data/division_comisaria_vecinal.csv"),
]

CRIME_FILE = next(
    (path for path in crime_candidates if path.exists()),
    None,
)

STATIONS_FILE = next(
    (path for path in station_candidates if path.exists()),
    None,
)

JURISDICTIONS_FILE = next(
    (path for path in jurisdiction_candidates if path.exists()),
    None,
)

if CRIME_FILE is None:
    raise FileNotFoundError(
        "No se encontró delitos_2023_caba_limpio.csv."
    )

if STATIONS_FILE is None:
    raise FileNotFoundError(
        "No se encontró el archivo de sedes policiales."
    )

if JURISDICTIONS_FILE is None:
    raise FileNotFoundError(
        "No se encontró division_comisaria_vecinal.csv."
    )

OUTPUT_FILE = (
    PROCESSED_DATA_DIR
    / "delitos_2023_caba_enriquecido_comisarias.csv"
)

KMEANS_REPORT_FILE = (
    TABLES_DIR
    / "evaluacion_kmeans_distancias.csv"
)

BANDS_REPORT_FILE = (
    TABLES_DIR
    / "reporte_franjas_distancia_comisaria.csv"
)

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Delitos: {CRIME_FILE}")
print(f"Sedes policiales: {STATIONS_FILE}")
print(f"Jurisdicciones: {JURISDICTIONS_FILE}")
print(f"Salida: {OUTPUT_FILE}")

## 3. Carga y validación inicial

Se controla que cada archivo contenga las columnas necesarias antes de comenzar el análisis.

In [ ]:
delitos = pd.read_csv(
    CRIME_FILE,
    low_memory=False,
)

sedes = pd.read_csv(
    STATIONS_FILE,
    low_memory=False,
)

jurisdicciones = pd.read_csv(
    JURISDICTIONS_FILE,
    low_memory=False,
)

required_crime_columns = {
    "id_mapa",
    "latitud",
    "longitud",
}

required_station_columns = {
    "nombre",
    "direccion",
    "geometry",
}

required_jurisdiction_columns = {
    "nombre",
    "division",
    "geometry",
}

missing_crime = sorted(
    required_crime_columns - set(delitos.columns)
)

missing_stations = sorted(
    required_station_columns - set(sedes.columns)
)

missing_jurisdictions = sorted(
    required_jurisdiction_columns - set(jurisdicciones.columns)
)

if missing_crime:
    raise ValueError(
        f"Faltan columnas en delitos: {missing_crime}"
    )

if missing_stations:
    raise ValueError(
        f"Faltan columnas en sedes: {missing_stations}"
    )

if missing_jurisdictions:
    raise ValueError(
        "Faltan columnas en jurisdicciones: "
        f"{missing_jurisdictions}"
    )

original_row_count = len(delitos)
original_column_count = delitos.shape[1]

print(
    f"Delitos: {len(delitos):,} filas y "
    f"{delitos.shape[1]} columnas"
)
print(f"Registros de sedes: {len(sedes):,}")
print(f"Jurisdicciones: {len(jurisdicciones):,}")

## 4. Validación de las coordenadas de los delitos

El Notebook 01 ya normalizó las coordenadas. Aquí se vuelve a comprobar que sean numéricas y que estén dentro de rangos amplios compatibles con CABA.

In [ ]:
delitos["latitud"] = pd.to_numeric(
    delitos["latitud"],
    errors="coerce",
)

delitos["longitud"] = pd.to_numeric(
    delitos["longitud"],
    errors="coerce",
)

invalid_crime_coordinates = (
    delitos["latitud"].isna()
    | delitos["longitud"].isna()
    | ~delitos["latitud"].between(-35, -34)
    | ~delitos["longitud"].between(-59, -58)
)

invalid_crime_count = int(
    invalid_crime_coordinates.sum()
)

print(
    "Coordenadas inválidas:",
    f"{invalid_crime_count:,}",
)

if invalid_crime_count:
    display(
        delitos.loc[
            invalid_crime_coordinates,
            [
                "id_mapa",
                "latitud",
                "longitud",
            ],
        ].head(20)
    )
    raise ValueError(
        "El archivo limpio contiene coordenadas inválidas."
    )

assert delitos["id_mapa"].is_unique
print("Todas las coordenadas de delitos son válidas.")

## 5. Preparación de las sedes policiales

La geometría de las sedes viene como:

```text
POINT (longitud latitud)
```

El archivo contiene 75 registros administrativos, pero algunas dependencias comunales y vecinales comparten exactamente el mismo domicilio. Para calcular distancia física se conservan **ubicaciones únicas**, evitando contar dos veces el mismo punto.

Cuando dos registros comparten coordenadas, se prioriza el nombre de la comisaría vecinal para construir una etiqueta breve. Si en una ubicación solo hay una comisaría comunal, se conserva esa denominación.

In [ ]:
point_pattern = (
    r"POINT\s*\(\s*"
    r"([+-]?\d+(?:\.\d+)?)\s+"
    r"([+-]?\d+(?:\.\d+)?)"
    r"\s*\)"
)

station_coordinates = (
    sedes["geometry"]
    .astype("string")
    .str.extract(point_pattern)
)

sedes["longitud_sede"] = pd.to_numeric(
    station_coordinates[0],
    errors="coerce",
)

sedes["latitud_sede"] = pd.to_numeric(
    station_coordinates[1],
    errors="coerce",
)

invalid_station_coordinates = (
    sedes["latitud_sede"].isna()
    | sedes["longitud_sede"].isna()
    | ~sedes["latitud_sede"].between(-35, -34)
    | ~sedes["longitud_sede"].between(-59, -58)
)

if invalid_station_coordinates.any():
    display(
        sedes.loc[
            invalid_station_coordinates,
            [
                "nombre",
                "direccion",
                "geometry",
            ],
        ]
    )
    raise ValueError(
        "Existen sedes con coordenadas inválidas."
    )

sedes["nombre"] = (
    sedes["nombre"]
    .astype("string")
    .str.strip()
)

sedes["direccion"] = (
    sedes["direccion"]
    .astype("string")
    .str.strip()
)

# Una sede vecinal se prioriza sobre una comunal cuando ambas
# comparten exactamente el mismo punto.
sedes["prioridad_vecinal"] = (
    sedes["nombre"]
    .str.contains("Vecinal", case=False, na=False)
    .astype(int)
)

sedes_unicas = (
    sedes
    .sort_values(
        [
            "latitud_sede",
            "longitud_sede",
            "prioridad_vecinal",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .drop_duplicates(
        subset=[
            "latitud_sede",
            "longitud_sede",
        ],
        keep="first",
    )
    .copy()
)

sedes_unicas["comisaria_referencia"] = (
    sedes_unicas["nombre"]
    + " — "
    + sedes_unicas["direccion"]
)

print(f"Registros administrativos: {len(sedes):,}")
print(f"Ubicaciones físicas únicas: {len(sedes_unicas):,}")

display(
    sedes_unicas[
        [
            "nombre",
            "direccion",
            "latitud_sede",
            "longitud_sede",
        ]
    ].head(10)
)

## 6. Asignación de la comisaría jurisdiccional

Los polígonos se interpretan mediante GeoPandas. Cada delito se cruza con las 45 jurisdicciones vecinales.

Para los pocos puntos situados apenas fuera de los polígonos por diferencias cartográficas, se asigna la jurisdicción más cercana. Esa corrección se informa en el notebook, pero no se agrega una columna técnica al CSV final porque no resulta útil para el panel.

In [ ]:
jurisdicciones_gdf = gpd.GeoDataFrame(
    jurisdicciones.drop(columns="geometry").copy(),
    geometry=gpd.GeoSeries.from_wkt(
        jurisdicciones["geometry"]
    ),
    crs="EPSG:4326",
)

if not jurisdicciones_gdf.geometry.is_valid.all():
    raise ValueError(
        "Hay polígonos jurisdiccionales inválidos."
    )

delitos_points_gdf = gpd.GeoDataFrame(
    delitos[
        [
            "id_mapa",
            "latitud",
            "longitud",
        ]
    ].copy(),
    geometry=gpd.points_from_xy(
        delitos["longitud"],
        delitos["latitud"],
    ),
    crs="EPSG:4326",
)

jurisdiction_join = gpd.sjoin(
    delitos_points_gdf,
    jurisdicciones_gdf[
        [
            "nombre",
            "geometry",
        ]
    ],
    how="left",
    predicate="intersects",
)

if jurisdiction_join["id_mapa"].duplicated().any():
    raise ValueError(
        "Algunos delitos intersectan más de una jurisdicción."
    )

missing_jurisdiction_mask = (
    jurisdiction_join["nombre"].isna()
)

missing_jurisdiction_count = int(
    missing_jurisdiction_mask.sum()
)

print(
    "Delitos sin intersección directa:",
    missing_jurisdiction_count,
)

if missing_jurisdiction_count:
    missing_points = (
        jurisdiction_join.loc[
            missing_jurisdiction_mask,
            [
                "id_mapa",
                "geometry",
            ],
        ]
        .to_crs("EPSG:32721")
    )

    jurisdictions_projected = (
        jurisdicciones_gdf[
            [
                "nombre",
                "geometry",
            ]
        ]
        .to_crs("EPSG:32721")
    )

    nearest_jurisdiction = gpd.sjoin_nearest(
        missing_points,
        jurisdictions_projected,
        how="left",
        distance_col="distancia_al_poligono_metros",
    )

    # Ante un empate exacto se conserva una sola jurisdicción.
    nearest_jurisdiction = (
        nearest_jurisdiction
        .sort_values(
            [
                "id_mapa",
                "distancia_al_poligono_metros",
            ]
        )
        .drop_duplicates(
            subset="id_mapa",
            keep="first",
        )
    )

    nearest_jurisdiction_map = (
        nearest_jurisdiction
        .set_index("id_mapa")["nombre"]
    )

    jurisdiction_join.loc[
        missing_jurisdiction_mask,
        "nombre",
    ] = (
        jurisdiction_join.loc[
            missing_jurisdiction_mask,
            "id_mapa",
        ]
        .map(nearest_jurisdiction_map)
    )

    print(
        "Distancia máxima usada para corregir esos puntos:",
        f"{nearest_jurisdiction['distancia_al_poligono_metros'].max():.2f} m",
    )

jurisdiction_map = (
    jurisdiction_join
    .set_index("id_mapa")["nombre"]
)

delitos["comisaria_jurisdiccional"] = (
    delitos["id_mapa"]
    .map(jurisdiction_map)
)

assert delitos["comisaria_jurisdiccional"].notna().all()

print("Jurisdicciones asignadas correctamente.")
display(
    delitos[
        [
            "id_mapa",
            "comisaria_jurisdiccional",
        ]
    ].head()
)

## 7. Cálculo de la sede policial más cercana

Se utiliza `BallTree` con la métrica de Haversine. Esta técnica permite buscar eficientemente el punto más próximo sobre la superficie terrestre.

La distancia obtenida es geodésica en línea recta. No representa recorrido caminando ni por calles.

In [ ]:
EARTH_RADIUS_METERS = 6_371_008.8

crime_coordinates_radians = np.radians(
    delitos[
        [
            "latitud",
            "longitud",
        ]
    ].to_numpy(dtype=float)
)

station_coordinates_radians = np.radians(
    sedes_unicas[
        [
            "latitud_sede",
            "longitud_sede",
        ]
    ].to_numpy(dtype=float)
)

station_tree = BallTree(
    station_coordinates_radians,
    metric="haversine",
)

distances_radians, nearest_indices = (
    station_tree.query(
        crime_coordinates_radians,
        k=1,
    )
)

nearest_indices = nearest_indices[:, 0]

nearest_stations = (
    sedes_unicas
    .iloc[nearest_indices]
    .reset_index(drop=True)
)

delitos["comisaria_mas_cercana"] = (
    nearest_stations["comisaria_referencia"]
    .to_numpy()
)

delitos["distancia_comisaria_metros"] = (
    np.rint(
        distances_radians[:, 0]
        * EARTH_RADIUS_METERS
    )
    .astype("int64")
)

assert delitos["comisaria_mas_cercana"].notna().all()
assert delitos["distancia_comisaria_metros"].ge(0).all()

display(
    delitos[
        [
            "id_mapa",
            "comisaria_jurisdiccional",
            "comisaria_mas_cercana",
            "distancia_comisaria_metros",
        ]
    ].head(10)
)

display(
    delitos["distancia_comisaria_metros"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .to_frame("distancia_en_metros")
)

## 8. Selección del número de franjas mediante K-Means

La distancia es una variable continua. Para convertirla en categorías útiles para el panel se aplica K-Means en una dimensión.

Se prueban soluciones entre 2 y 8 grupos y se comparan:

- **inercia:** variabilidad interna de los grupos; se utiliza para el método del codo;
- **coeficiente de silueta:** mide separación entre grupos.

### Consideración importante

La mejor silueta no necesariamente produce la segmentación más útil para una aplicación. Una división en dos grupos puede separar mejor los datos, pero sería demasiado general para distinguir entorno inmediato, cercanía intermedia y zonas alejadas.

Para detectar el codo se calcula la distancia de cada punto de la curva a la recta que une el primer y el último valor. Es una regla reproducible, no una elección visual arbitraria.

In [ ]:
distance_values = (
    delitos["distancia_comisaria_metros"]
    .to_numpy()
)

# Las distancias están redondeadas a metros enteros.
# Para acelerar K-Means, se agrupan valores iguales y se usa
# su frecuencia como peso. El resultado equivale a entrenar
# sobre todas las filas repetidas.
unique_distances, distance_frequencies = np.unique(
    distance_values,
    return_counts=True,
)

X_unique = unique_distances.reshape(-1, 1)

sample_size = min(
    12_000,
    len(delitos),
)

silhouette_sample = (
    delitos[
        [
            "distancia_comisaria_metros",
        ]
    ]
    .sample(
        n=sample_size,
        random_state=42,
    )
    .to_numpy()
)

evaluation_rows = []
fitted_models = {}

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10,
    )

    model.fit(
        X_unique,
        sample_weight=distance_frequencies,
    )

    sample_labels = model.predict(
        silhouette_sample
    )

    silhouette = silhouette_score(
        silhouette_sample,
        sample_labels,
    )

    fitted_models[k] = model

    evaluation_rows.append(
        {
            "k": k,
            "inercia": model.inertia_,
            "silueta": silhouette,
        }
    )

kmeans_evaluation = pd.DataFrame(
    evaluation_rows
)

kmeans_evaluation["reduccion_inercia_pct"] = (
    kmeans_evaluation["inercia"]
    .pct_change()
    .mul(-100)
    .round(2)
)

# Detección geométrica del codo.
x = (
    kmeans_evaluation["k"]
    .to_numpy(dtype=float)
)

y = (
    kmeans_evaluation["inercia"]
    .to_numpy(dtype=float)
)

x_normalized = (
    (x - x.min())
    / (x.max() - x.min())
)

y_normalized = (
    (y - y.min())
    / (y.max() - y.min())
)

distance_to_line = (
    (1 - x_normalized)
    - y_normalized
) / np.sqrt(2)

kmeans_evaluation["distancia_recta_codo"] = (
    distance_to_line
)

ELBOW_K = int(
    kmeans_evaluation.loc[
        kmeans_evaluation[
            "distancia_recta_codo"
        ].idxmax(),
        "k",
    ]
)

display(kmeans_evaluation.round(4))

print(
    "Cantidad de grupos sugerida por el codo:",
    ELBOW_K,
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    kmeans_evaluation["k"],
    kmeans_evaluation["inercia"],
    marker="o",
)
plt.axvline(
    ELBOW_K,
    linestyle="--",
    label=f"Codo estimado: k = {ELBOW_K}",
)
plt.title("Método del codo para las distancias")
plt.xlabel("Cantidad de grupos (k)")
plt.ylabel("Inercia")
plt.xticks(kmeans_evaluation["k"])
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "metodo_codo_distancias.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    kmeans_evaluation["k"],
    kmeans_evaluation["silueta"],
    marker="o",
)
plt.title("Coeficiente de silueta por cantidad de grupos")
plt.xlabel("Cantidad de grupos (k)")
plt.ylabel("Coeficiente de silueta")
plt.xticks(kmeans_evaluation["k"])
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "silueta_distancias.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 9. Interpretación de la solución con cuatro grupos

El método del codo selecciona `k = 4`. La silueta suele favorecer una solución más simple de dos grupos, pero esa alternativa no resulta suficientemente informativa para un panel.

Con cuatro grupos se obtienen segmentos distinguibles y de tamaño útil, sin crear categorías residuales casi vacías.

In [ ]:
selected_model = fitted_models[ELBOW_K]

cluster_centers = np.sort(
    selected_model.cluster_centers_
    .ravel()
)

statistical_boundaries = (
    cluster_centers[:-1]
    + cluster_centers[1:]
) / 2

cluster_interpretation = pd.DataFrame(
    {
        "grupo": range(
            1,
            len(cluster_centers) + 1,
        ),
        "centro_metros": (
            cluster_centers.round(2)
        ),
    }
)

display(cluster_interpretation)

print(
    "Límites estadísticos aproximados:",
    np.round(
        statistical_boundaries,
        2,
    ),
)

## 10. Conversión de los límites estadísticos en franjas para el panel

Los límites obtenidos por K-Means son aproximadamente:

```text
493 m, 867 m y 1.352 m
```

Para una interfaz resulta poco útil mostrar cortes como `493` o `867`. Se adoptan valores cercanos, más fáciles de leer y recordar:

| Franja | Corte utilizado |
|---|---:|
| Entorno inmediato | 0 a 500 m |
| Cercana | 501 a 900 m |
| Intermedia | 901 a 1.350 m |
| Alejada | más de 1.350 m |

Los ajustes respecto de los límites estadísticos son pequeños y preservan la estructura de los cuatro grupos.

In [ ]:
distance_bins = [
    -1,
    500,
    900,
    1350,
    np.inf,
]

distance_labels = [
    "ENTORNO INMEDIATO (0-500 m)",
    "CERCANA (501-900 m)",
    "INTERMEDIA (901-1350 m)",
    "ALEJADA (MAS DE 1350 m)",
]

delitos["franja_distancia_comisaria"] = pd.cut(
    delitos["distancia_comisaria_metros"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=True,
    right=True,
    ordered=True,
)

assert delitos[
    "franja_distancia_comisaria"
].notna().all()

band_report = (
    delitos[
        "franja_distancia_comisaria"
    ]
    .value_counts(sort=False)
    .rename_axis(
        "franja_distancia_comisaria"
    )
    .reset_index(name="cantidad_delitos")
)

band_report["porcentaje"] = (
    band_report["cantidad_delitos"]
    .div(len(delitos))
    .mul(100)
    .round(2)
)

band_report["porcentaje_acumulado"] = (
    band_report["cantidad_delitos"]
    .cumsum()
    .div(len(delitos))
    .mul(100)
    .round(2)
)

display(band_report)

plt.figure(figsize=(11, 5))
plt.bar(
    band_report[
        "franja_distancia_comisaria"
    ].astype(str),
    band_report["cantidad_delitos"],
)
plt.title(
    "Delitos por franja de distancia "
    "a la sede policial más cercana"
)
plt.xlabel("Franja de distancia")
plt.ylabel("Cantidad de delitos")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR
    / "distribucion_franjas_distancia.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 11. Controles del dataset enriquecido

Se verifica que:

- no se haya perdido ni duplicado ningún delito;
- las cuatro columnas nuevas estén completas;
- las distancias sean no negativas;
- las franjas incluyan todos los registros;
- el dataset final agregue exactamente cuatro columnas.

In [ ]:
new_columns = [
    "comisaria_jurisdiccional",
    "comisaria_mas_cercana",
    "distancia_comisaria_metros",
    "franja_distancia_comisaria",
]

assert len(delitos) == original_row_count
assert delitos["id_mapa"].is_unique
assert delitos.shape[1] == original_column_count + 4
assert delitos[new_columns].notna().all().all()
assert delitos[
    "distancia_comisaria_metros"
].ge(0).all()

print("Todos los controles fueron superados.")
print(
    f"Filas finales: {len(delitos):,}"
)
print(
    f"Columnas finales: {delitos.shape[1]}"
)

## 12. Exportación

Se exportan:

1. el dataset enriquecido;
2. la evaluación de K-Means;
3. el reporte de las franjas;
4. los gráficos del codo, la silueta y la distribución final.

In [ ]:
delitos.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

kmeans_evaluation.to_csv(
    KMEANS_REPORT_FILE,
    index=False,
    encoding="utf-8-sig",
)

band_report.to_csv(
    BANDS_REPORT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Archivos exportados:")
print(f" - Dataset: {OUTPUT_FILE}")
print(f" - Evaluación K-Means: {KMEANS_REPORT_FILE}")
print(f" - Reporte de franjas: {BANDS_REPORT_FILE}")
print(
    " - Gráfico del codo:",
    FIGURES_DIR / "metodo_codo_distancias.png",
)
print(
    " - Gráfico de silueta:",
    FIGURES_DIR / "silueta_distancias.png",
)
print(
    " - Gráfico de franjas:",
    FIGURES_DIR
    / "distribucion_franjas_distancia.png",
)

## 13. Conclusión metodológica

Se combinaron ambos datasets de comisarías porque aportan información complementaria:

- las sedes permiten medir proximidad física;
- los polígonos permiten asignar jurisdicción.

K-Means se utilizó para evitar seleccionar arbitrariamente la cantidad de franjas. El método del codo indicó cuatro grupos. Sus límites estadísticos se redondearon a valores operativos para que las categorías fueran claras en una aplicación.

La variable final `franja_distancia_comisaria` no debe interpretarse automáticamente como nivel de riesgo. Es una dimensión descriptiva que puede cruzarse con:

- franja horaria;
- barrio o comuna;
- jurisdicción;
- uso de arma;
- uso de moto;
- tipo o subtipo del hecho.

De esta manera, el panel puede mostrar patrones sin atribuir causalidad a la cercanía de una comisaría.